# Gurobi Tutorial

## Problem definition
### The Multi-Period Blending Problem  
A manufacturing plant produces two types of metal alloys **Standard** and **Premium**. These are produced by blending two raw materials: **Tin** and **Iron**. The planning horizon is three months.

You are given the following data:

1. Demand:

|**Month**|1|2|3|
|---|---|---|---|
|**Standard**|100|150|200|
|**Premium**|50|70|90|

2. Raw Material Costs(per unit):

|**Month**|1|2|3|
|---|---|---|---|
|**Tin**|10|12|11|
|**Iron**|15|14|16|

3. Holding costs:
- Unused raw materials cost $1 per unit per month to store.
- Unsold finished products cost $2 per unit per month to store.
- Initial inventories for all materials and products are 0.

4. capacity:
- The factory can produce a maximum total of 250 units of finished products per month.

5. Blending Specifications:
- The **Standard** product must contain at least 40% of **Tin**.
- The **Premium** product must contain at least 60% of **Iron**.
- There is no mass loss during production (1 unit of raw material yields 1 unit of finished product).

The goal is to determine the optimal purchasing, production, and inventory strategy to minimize the total cost while meeting all demands


### Mathematical Formulation

#### Sets and Indices

* $T = \{1, 2, 3\}$: Set of time periods (months), indexed by $t$.
* $R = \{\text{A}, \text{B}\}$: Set of raw materials, indexed by $r$ (A: Tin, B: Iron). 
* $P = \{\text{Std}, \text{Prem}\}$: Set of products, indexed by $p$.

#### Parameters

* $D_{p,t}$: Demand for product $p$ in month $t$.
* $C_{r,t}$: Purchase cost of raw material $r$ in month $t$.
* $H^R = 1$: Holding cost per unit of raw material.
* $H^P = 2$: Holding cost per unit of finished product.
* $K = 250$: Monthly production capacity.

#### Decision Variables (Continuous, $\ge 0$)

* $x_{r,t}$: Units of raw material $r$ purchased in month $t$.
* $y_{r,p,t}$: Units of raw material $r$ used to produce product $p$ in month $t$.
* $z_{p,t}$: Total units of product $p$ produced in month $t$.
* $I^R_{r,t}$: Inventory of raw material $r$ at the end of month $t$.
* $I^P_{p,t}$: Inventory of product $p$ at the end of month $t$.

#### Objective Function

Minimize the sum of purchasing costs and holding costs:


$$ \min \sum_{t \in T} \left( \sum_{r \in R} C_{r,t} x_{r,t} + \sum_{r \in R} H^R I^R_{r,t} + \sum_{p \in P} H^P I^P_{p,t} \right) $$

#### Constraints

**1. Mass Balance (Production):**
The total amount of a product manufactured equals the sum of the raw materials used to make it.


$$ z_{p,t} = \sum_{r \in R} y_{r,p,t} \quad \forall p \in P, \forall t \in T $$

**2. Production Capacity:**


$$ \sum_{p \in P} z_{p,t} \le K \quad \forall t \in T $$

**3. Blending Requirements:**


$$ y_{\text{A}, \text{Std}, t} \ge 0.4 \cdot z_{\text{Std}, t} \quad \forall t \in T $$

$$ y_{\text{B}, \text{Prem}, t} \ge 0.6 \cdot z_{\text{Prem}, t} \quad \forall t \in T $$

**4. Inventory Balance for Raw Materials:**
Inventory at the end of $t$ equals previous inventory plus purchases minus usage. (Assume $I^R_{r,0} = 0$).


$$ I^R_{r,t} = I^R_{r,t-1} + x_{r,t} - \sum_{p \in P} y_{r,p,t} \quad \forall r \in R, \forall t \in T $$

**5. Inventory Balance for Finished Products:**
Inventory at the end of $t$ equals previous inventory plus production minus demand. (Assume $I^P_{p,0} = 0$).


$$ I^P_{p,t} = I^P_{p,t-1} + z_{p,t} - D_{p,t} \quad \forall p \in P, \forall t \in T $$

# Gurobi API - getting started

In [1]:
%load_ext autoreload
%autoreload 2   
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import scipy.sparse as sp

In [2]:
# Create a new model
# m = gp.Model("mip1")

# Gurobi python API also supports sparse matrices, which is useful for large problems.
# If you wanna use matrix:
# m = gp.Model("matrix1")

See [API ref](https://support.gurobi.com/hc/en-us/articles/17307437899025-Tutorial-Getting-Started-with-the-Gurobi-Python-API-using-dictionaries) for detail explanation.


In [2]:
# 1. Define Sets and Parameters
T = [1, 2, 3]
R = ['A', 'B']
P = ['Std', 'Prem']

demand = {
    ('Std', 1): 100, ('Std', 2): 150, ('Std', 3): 200,
    ('Prem', 1): 50,  ('Prem', 2): 70,  ('Prem', 3): 90
}

cost = {
    ('A', 1): 10, ('A', 2): 12, ('A', 3): 11,
    ('B', 1): 15, ('B', 2): 14, ('B', 3): 16
}

H_R = 1
H_P = 2
K = 250

# 2. Initialize Model
m = gp.Model("MultiPeriodBlending")

# 3. Create Variables
# Using m.addVars creates a gurobipy.tupledict, which allows for efficient slicing and summing
x = m.addVars(R, T, vtype=GRB.CONTINUOUS, name="buy")
y = m.addVars(R, P, T, vtype=GRB.CONTINUOUS, name="blend")
z = m.addVars(P, T, vtype=GRB.CONTINUOUS, name="produce")
I_R = m.addVars(R, T, vtype=GRB.CONTINUOUS, name="inv_raw")
I_P = m.addVars(P, T, vtype=GRB.CONTINUOUS, name="inv_prod")

# 4. Set Objective
# Minimize total cost: Purchasing + Raw material holding + Product holding
m.setObjective(
    gp.quicksum(cost[r, t]*x[r, t] for r in R for t in T) +
    gp.quicksum(H_R*I_R[r, t] for r in R for t in T) + 
    gp.quicksum(H_P*I_P[p, t] for p in P for t in T),
    GRB.MINIMIZE
)


# 5. Add Constraints

# Mass Balance: Production equals sum of blended materials.
# The '*' acts as a wildcard to sum over the raw material index (R).
m.addConstrs(
    (z[p, t] == y.sum('*', p, t) for p in P for t in T), 
    name="mass_balance"
)

# Production Capacity: Total blended materials cannot exceed K.
m.addConstrs(
    (z.sum('*', t) <= K for t in T),
    name="capacity"
)

# Blending Requirements
m.addConstrs(
    (y['A', 'Std', t] >= 0.4 * z['Std', t] for t in T), 
    name="blend_std"
)
m.addConstrs(
    (y['B', 'Prem', t] >= 0.6 * z['Prem', t] for t in T), 
    name="blend_prem"
)

# Inventory Balance for Raw Materials
for r in R:
    for t in T:
        inv_prev = I_R[r, t-1] if t > 1 else 0
        m.addConstr(
            (inv_prev + x[r, t] - y.sum(r, '*', t) == I_R[r, t]),
            name=f"inv_raw_bal_{r}_{t}"
        )

# Inventory Balance for Finished Products
for p in P:
    for t in T:
        inv_prev = I_P[p, t-1] if t > 1 else 0
        m.addConstr(
            I_P[p, t] == inv_prev + z[p, t] - demand[p, t],
            name=f"inv_prod_bal_{p}_{t}"
        )



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2819691
Academic license 2819691 - for non-commercial use only - registered to u5___@ecs.osaka-u.ac.jp


In [3]:
# 6. Optimize
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: 13th Gen Intel(R) Core(TM) i7-13700H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 20 logical processors, using up to 20 threads

Academic license 2819691 - for non-commercial use only - registered to u5___@ecs.osaka-u.ac.jp
Optimize a model with 27 rows, 36 columns and 80 nonzeros (Min)
Model fingerprint: 0x2b49de2a
Model has 18 linear objective coefficients
Coefficient statistics:
  Matrix range     [4e-01, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 2e+02]

Presolve removed 20 rows and 26 columns
Presolve time: 0.03s
Presolved: 7 rows, 10 columns, 23 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2065620e+03   8.806638e+01   0.000000e+00      0s
       5    7.6680000e+03   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.04 seconds (0.00 work units)
Optimal

In [4]:
if m.status == GRB.OPTIMAL:
    print("-" * 30)
    print(f"Optimal Total Cost: ${m.objVal:.2f}")
    print("-" * 30)

    # Iterate through all variables and print those with non-zero values
    for v in m.getVars():
        if v.x > 1e-6:  # Tolerance threshold for floating point precision
            print(f"{v.varName}: {v.x}")
else:
    print("Optimization did not converge to an optimal solution.")

------------------------------
Optimal Total Cost: $7668.00
------------------------------
buy[A,1]: 314.0
buy[A,3]: 220.0
buy[B,1]: 30.0
buy[B,2]: 96.0
blend[A,Std,1]: 110.0
blend[A,Std,2]: 140.0
blend[A,Std,3]: 200.0
blend[A,Prem,1]: 20.0
blend[A,Prem,2]: 44.0
blend[A,Prem,3]: 20.0
blend[B,Prem,1]: 30.0
blend[B,Prem,2]: 66.0
blend[B,Prem,3]: 30.0
produce[Std,1]: 110.0
produce[Std,2]: 140.0
produce[Std,3]: 200.0
produce[Prem,1]: 50.0
produce[Prem,2]: 110.0
produce[Prem,3]: 50.0
inv_raw[A,1]: 184.0
inv_raw[B,2]: 30.0
inv_prod[Std,1]: 10.0
inv_prod[Prem,2]: 40.0


Well done!

# Decoding problem 

### Mathematical Formulation for Minimum Weight Decoding

The primary challenge in formulating the decoding problem for a standard Mixed Integer Linear Programming (MILP) solver like Gurobi is the modulo-2 arithmetic required by the parity check equation $He \equiv \sigma \pmod 2$. MILP solvers operate over the field of real numbers, meaning this constraint must be linearized.

To linearize the modulo-2 constraint, an auxiliary integer variable vector $z$ is introduced. The operation modulo 2 implies that the difference between the standard integer sum $\sum_j H_{ij} e_j$ and the syndrome $\sigma_i$ must be an even integer.

#### Variables

* $e_j \in \{0, 1\}$ for $j \in \{1, \dots, n\}$: Binary decision variables representing the error vector.
* $z_i \in \mathbb{Z}_{\ge 0}$ for $i \in \{1, \dots, m\}$: Auxiliary non-negative integer variables representing the quotient for the modulo-2 operation.

#### Objective Function

Minimize the total weight of the predicted error vector:


$$ \min \sum_{j=1}^{n} w_j e_j $$

#### Constraints

For each parity check constraint $i$ (where $i \in \{1, \dots, m\}$), the sum of the incident errors must equal the syndrome bit plus some even integer:


$$ \sum_{j=1}^{n} H_{i,j} e_j - 2z_i = \sigma_i \quad \forall i \in \{1, \dots, m\} $$

### Gurobi Python Implementation

Below is the Python implementation of this formulation. The parity check matrix $H$ is typically highly sparse in quantum error correction (e.g., LDPC codes or color codes). While the example uses standard dense arrays for simplicity, Gurobi handles sparse matrix constructions efficiently if you pass `scipy.sparse` matrices or construct the sums using only the non-zero indices.

In [8]:
def solve_mwd(H: np.ndarray, syndrome: np.ndarray, weights: np.ndarray):
    """
    Solves the Minimum Weight Decoding problem using Gurobi.
    
    Parameters:
    H (np.ndarray): Parity check matrix of shape (m, n) with elements in {0, 1}.
    syndrome (np.ndarray): Syndrome vector of shape (m,) with elements in {0, 1}.
    weights (np.ndarray): Error weight vector of shape (n,).
    
    Returns:
    np.ndarray: Optimal error vector e.
    """
    m_checks, n_errors = H.shape
    
    # Initialize Model
    model = gp.Model("MinimumWeightDecoding")
    
    # Disable console output for faster execution in loops, if desired
    # model.Params.LogToConsole = 0
    
    # Variables
    # e: Binary error vector
    e = model.addVars(n_errors, vtype=GRB.BINARY, name="e")
    
    # z: Auxiliary integer variables for modulo 2
    # Upper bound can be restricted to floor(max_row_weight / 2) to tighten the relaxation
    max_z = (np.max(np.sum(H, axis=1)) // 2)
    z = model.addVars(m_checks, lb=0, ub=max_z, vtype=GRB.INTEGER, name="z")
    
    # Objective: Minimize w^T * e
    obj_expr = gp.quicksum(weights[j] * e[j] for j in range(n_errors))
    model.setObjective(obj_expr, GRB.MINIMIZE)
    
    # Constraints: H * e = syndrome + 2 * z
    for i in range(m_checks):
        # Find non-zero indices in row i to optimize expression building
        non_zero_indices = np.where(H[i, :] == 1)[0]
        
        row_sum = gp.quicksum(e[j] for j in non_zero_indices)
        model.addConstr(row_sum - 2 * z[i] == syndrome[i], name=f"parity_{i}")
        
    # Optimize
    model.optimize()
    
    # Extract results
    if model.status == GRB.OPTIMAL:
        e_opt = np.array([e[j].X for j in range(n_errors)])
        # Rounding to handle potential floating point inaccuracies from the solver
        return np.round(e_opt).astype(int)
    else:
        raise RuntimeError("Optimal solution not found. Solver status: " + str(model.status))


In [9]:
# Example Execution
if __name__ == "__main__":
    # Small Repetition Code Example
    H_example = np.array([
        [1, 1, 0],
        [0, 1, 1]
    ])
    
    # Suppose a single error on qubit 1 (middle qubit)
    syndrome_example = np.array([1, 1]) 
    
    # Weights derived from p = 0.1: log((1-0.1)/0.1) ~ 2.19
    weights_example = np.array([2.19, 2.19, 2.19])
    
    optimal_error = solve_mwd(H_example, syndrome_example, weights_example)
    
    print("Given Syndrome:", syndrome_example)
    print("Optimal Error Vector:", optimal_error)


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: 13th Gen Intel(R) Core(TM) i7-13700H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 2 rows, 5 columns and 6 nonzeros (Min)
Model fingerprint: 0xfabedb93
Model has 3 linear objective coefficients
Variable types: 0 continuous, 5 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [2e+00, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 2 rows and 5 columns
Presolve time: 0.01s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.04 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)

Solution count 1: 2.19 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.190000000000e+00, best bound 2.190000000000e+00, gap 0.0000%
Given Syndrome: [1 1]
Optimal Error Vector